In [2]:
from functools import partial

import astroplan as ap
import astropy.units as u
import numpy as np
import pandas as pd

from astropaul.database import database_connection, html_path
import astropaul.html as html
import astropaul.targetlistcreator as tlc
import astropaul.gemini as gemini

%load_ext autoreload
%autoreload 2


## For next proposal:
* use RA limits instead of calculating observability
* try to match the rounding process they use
* make sure to update our records of prior Gemini observations before running the target list

In [3]:
site_names = [
    "Gemini South",
    "Gemini North", # since targets are filtered if they appear in a previous list, put North after South to give South more work
]
sessions = {}
for site in site_names:
    session = tlc.ObservingSession(ap.Observer.at_site(site))
    for month in range(8, 13):
        session.add_full_day(f"2026-{month:02d}-01")
        session.add_full_day(f"2026-{month:02d}-11")
        session.add_full_day(f"2026-{month:02d}-21")
    sessions[site] = session

In [ ]:
name = "Gemini 2026B Proposal"
html_dir = html_path() / name
html.clear_directory(html_dir)

lower_mag_limit = 12
upper_mag_limit = 15

# mag>12 no gemini AND not resolved

exposure_time_field = "gemini_iq70_speckle_program_time"

###########################################################################################################
# LOOK AT PROPOSAL NEXT TIME - IT SPECIFIES RA AND DEC ALLOWED RANGES RATHER THAN CALCULATING OBSERVABILITY
###########################################################################################################
targets_pit_will_reject = {
    "TIC 219469945",  # PIT claimed not visible
    "TIC 219006972",  # PIT claimed not visible
    "TIC 414475823",  # PIT claimed not visible
    "TIC 285655079",  # PIT claimed not visible
    "TIC 250119205",  # PIT claimed not visible
    "TIC 308731030",  # PIT claimed not visible
    "TIC 37376063",  # existing data in the GOA
    "TIC 52856877",  # existing data in the GOA
}

common_steps = [
    tlc.add_targets,
    tlc.add_lists,
    tlc.ancillary_data_from_tess,
    partial(tlc.filter_targets, criteria=lambda df: df["List Kostov EBs"]),
    partial(tlc.add_database_table, table_name="ZorroAlopeke Observations"),
    partial(tlc.filter_targets, criteria=lambda df: df["Num ZorroAlopeke Observations"] == 0),
    partial(tlc.filter_targets, inverse=True, criteria=lambda df: df["Target Name"].isin(targets_pit_will_reject)),
    gemini.add_gemini_speckle_params,
]

list_steps = {
    "HQUS": [
        partial(tlc.filter_targets, criteria=lambda df: df["List HQND 2025-09-30"]),
    ],
    "Priority 1 Faint": [
        partial(tlc.filter_targets, criteria=lambda df: df["List Kostov 2022"]),
        partial(tlc.filter_targets, criteria=lambda df: df["Vmag"] > lower_mag_limit),
        partial(tlc.add_database_table, table_name="Speckle Detections"),
        partial(tlc.filter_targets, criteria=lambda df: df["Num Speckle Detections"] == 0),
    ],
    "Priority 2 Faint": [
        partial(tlc.filter_targets, criteria=lambda df: df["List Kostov 2023"]),
        partial(tlc.filter_targets, criteria=lambda df: df["Vmag"] > lower_mag_limit),
        partial(tlc.filter_targets, criteria=lambda df: df["Vmag"] < upper_mag_limit),
        partial(tlc.add_database_table, table_name="DSSI Observations"),
        partial(tlc.filter_targets, criteria=lambda df: df["Num DSSI Observations"] == 0),
    ],
}

site_specific_steps = [
    partial(tlc.add_observability, observability_threshold=(35 * u.deg, 80 * u.deg), time_resolution=60 * u.min),
    partial(tlc.filter_targets, criteria=lambda df: df["Observable Any Night"]),
]

targets_already_added = set()
lists = {}
with database_connection() as conn:
    for list_prefix, steps in list_steps.items():
        for site_name, session in sessions.items():
            list_name = f"{list_prefix} {site_name}"
            creator = tlc.TargetListCreator(name=list_name, connection=conn)
            this_steps = (
                common_steps
                + steps
                + [partial(tlc.filter_targets, inverse=True, criteria=lambda df: df["Target Name"].isin(targets_already_added))]
                + site_specific_steps
            )
            tl = creator.calculate(steps=this_steps, observing_session=session, verbose=False)
            tl.target_list["Group Category"] = list_prefix
            tl.target_list["Group Name"] = list_name
            tl.target_list["Observatory"] = pd.cut(
                tl.target_list["Dec"], bins=[-90, -25, 25, 90], labels=["South", "Both", "North"]
            )
            lists[list_name] = tl
            targets_already_added.update(tl.target_list["Target Name"])
            print(f"{len(tl.target_list):2d}   {tl.target_list[exposure_time_field].sum()/3600:4.1f}   {list_name}")


unified_list = tlc.TargetList.union(list(lists.values()), name="Gemini 2026B Proposal")

# targets_to_skip = ["TIC 52856877", "TIC 50198590"]  # , "TIC 244279814"] # PIT disagreed with me that these are observable
# unified_list.target_list = unified_list.target_list[~unified_list.target_list["Target Name"].isin(targets_to_skip)]
proposal_targets = unified_list.target_list

output_list = proposal_targets[["Target Name", "RA", "Dec", "Vmag", exposure_time_field]]
output_list.columns = ["Name", "RAJ2000", "DecJ2000", "V", "Exposure Time"]
output_list.to_csv("Gemini 2026B Targets.csv", index=False)
print(f"{len(proposal_targets)} targets, {proposal_targets[exposure_time_field].sum() / 3600:.2f} hours")

# print(unified_list.summarize())
html.render_observing_pages(unified_list, None, {}, html_dir)

10    2.3   HQUS Gemini South
 6    1.4   HQUS Gemini North
12    3.3   Priority 1 Faint Gemini South
 7    1.9   Priority 1 Faint Gemini North
21    6.1   Priority 2 Faint Gemini South
12    3.1   Priority 2 Faint Gemini North
68 targets, 18.14 hours


After going through the entire proposal process, the PIT software had problems with specific targets.  Remove them. 

* Start the PIT and open the file `Gemini 2026B Proposal BASE.xml`
  * This base file has investigators, title, and other fields already populated
* Import the target list created above into the PIT
* In targets tab, sort by Dec
* Select & Copy all targets Dec < -10
* Switch to Observations tab and Paste
* Specify weather conditions and Zorro speckle (South)
* Select & Copy all targets Dec > -10
* Switch to Observations tab and Paste
* Specify weather conditions and `Alopeke speckle (North)
* Do a Save As in the PIT, to the file `Gemini 2026B Proposal TARGETS.xml`
* The below code will add observing times for each target to the xml file

In [11]:
# edit the xml file saved by the PIT and add observation times
import xml.etree.ElementTree as ET

filename = "Gemini 2026B Proposal"

# first, get observation times for all targets (in hours, because that's all PIT accepts)
target_times = {
    str(name): (iq70 / 3600, iq85 / 3600)
    for _, (name, iq70, iq85) in proposal_targets[
        ["Target Name", "gemini_iq70_speckle_program_time", "gemini_iq85_speckle_program_time"]
    ].iterrows()
}

# for each weather condition:
# get the mapping of the nickname used in xml file to the TIC name
# then walk through each observation entry and add observing time to it for the appropriate conditions
tot = {"iq70": 0.0, "iq85": 0.0}
for index, name in enumerate(["iq70", "iq85"]):
    tree = ET.parse(f"{filename} TARGETS.xml")
    root = tree.getroot()
    target_map = {}
    for target in root.findall("targets/sidereal"):
        designator = target.attrib["id"]
        nickname = target.find("name").text
        target_map[designator] = nickname

    for observation in root.findall("observations/observation"):
        target_name = target_map[observation.attrib["target"]]
        target_time = target_times[target_name][index]
        tot[name] += target_time
        target_time_text = f"{target_time:.3f}"
        progTime = ET.Element("progTime")
        progTime.set("units", "s")
        progTime.text = target_time_text
        observation.append(progTime)
        time = ET.Element("time")
        time.set("units", "s")
        time.text = target_time_text
        observation.append(time)
    tree.write(f"{filename} {name}.xml")

tot

{'iq70': 18.143416426666665, 'iq85': 29.568024853333323}

* Use the above xml editing code to apply exposure times
* Make copy of the iq70 file with a band3 suffix
* Open the iq85 file and copy all the `<observation>` nodes
* Rename `Band 1/2` to `Band 3` for all nodes
* Paste these new band 3 nodes below the band 1/2 nodes in the new band3 file created above

In [12]:
observatories = ["North", "Both", "South"]
sub_lists = proposal_targets["Group Category"].unique()


def Count(x):
    return len(x)


def Hours(x):
    return np.sum(x) / 3600


# proposal_targets["Group Category"] = [
#     f"Faint ($V > {lower_mag_limit}$)" if "Faint" in name else name for name in proposal_targets["Group Category"]
# ]

grid = (
    proposal_targets.pivot_table(
        index="Group Category",
        columns="Observatory",
        values="gemini_iq70_speckle_program_time",
        # values="gemini_iq85_speckle_program_time",
        aggfunc=[Count, Hours],
        fill_value=0,
        observed=False,
    )
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1, level=0)
)
grid.loc["Total"] = grid.sum()

formats = {
    ("South", "Count"): "{:.0f}",
    ("Both", "Count"): "{:.0f}",
    ("North", "Count"): "{:.0f}",
    ("South", "Hours"): "{:.2f}",
    ("Both", "Hours"): "{:.2f}",
    ("North", "Hours"): "{:.2f}",
}

foo = grid.style.format(formats)
print(foo.to_latex())
grid

\begin{tabular}{lrrrrrr}
Observatory & \multicolumn{2}{r}{South} & \multicolumn{2}{r}{Both} & \multicolumn{2}{r}{North} \\
 & Count & Hours & Count & Hours & Count & Hours \\
Group Category &  &  &  &  &  &  \\
HQUS & 8 & 1.93 & 2 & 0.41 & 6 & 1.39 \\
Priority 1 Faint & 3 & 0.82 & 9 & 2.47 & 7 & 1.89 \\
Priority 2 Faint & 12 & 3.55 & 10 & 2.80 & 11 & 2.88 \\
Total & 23 & 6.30 & 21 & 5.68 & 24 & 6.17 \\
\end{tabular}



Observatory      South            Both           North          
                 Count     Hours Count     Hours Count     Hours
Group Category                                                  
HQUS               8.0  1.928693   2.0  0.408398   6.0  1.393823
Priority 1 Faint   3.0  0.823383   9.0  2.470150   7.0  1.893123
Priority 2 Faint  12.0  3.546477  10.0  2.800821  11.0  2.878548
Total             23.0  6.298553  21.0  5.679369  24.0  6.165494

* Paste the above table into the Overleaf for the experimental design
* For Phase 3, I generated a version of the above table for iq85 conditions, and Jimmy used these to enter a minimum Band 3 request, either from just the HQUS or that plus Priority 1 (Kostov 22). ~~I just added the HQUS targets and their IQ85 exposure times, manually.~~
  * Ideally, my code could create separate Phase 1/2 and Phase 3 lists using different IQ70/85 exposure times, but I didn't have time for that this time.